In [2]:
import pandas as pd
import numpy as np

## Automatic Data Alignment

When performing arithmetic between pandas objects with different indexes, pandas **automatically aligns** values by their labels before computing.

### Rules

- The result index is the **union** of the two indexes.
- Labels present in only one object produce `NaN` in the result.
- `NaN` propagates through further arithmetic operations.

### On a Series

Alignment is performed on the index labels.

### On a DataFrame

Alignment is performed on **both** row and column labels simultaneously.

- A row label found in only one DataFrame → entire row becomes `NaN`.
- A column label found in only one DataFrame → entire column becomes `NaN`.
- If no row **or** column labels overlap at all → result is entirely `NaN`.

## Key Points

- Arithmetic between pandas objects aligns on labels, not positions.
- The result index/columns is the union of both objects.
- Non-overlapping labels produce `NaN`.
- `NaN` propagates in all further arithmetic.

In [3]:
s1 = pd.Series([7.3, -2.5, 3.4, 1.5], index=["a", "c", "d", "e"])
s2 = pd.Series([-2.1, 3.6, -1.5, 4, 3.1], index=["a", "c", "e", "f", "g"])

print(s1)
print()
print(s2)

a    7.3
c   -2.5
d    3.4
e    1.5
dtype: float64

a   -2.1
c    3.6
e   -1.5
f    4.0
g    3.1
dtype: float64


In [4]:
s1 + s2

a    5.2
c    1.1
d    NaN
e    0.0
f    NaN
g    NaN
dtype: float64

In [5]:
df1 = pd.DataFrame(
    np.arange(9.).reshape((3, 3)),
    columns=list("bcd"),
    index=["Ohio", "Texas", "Colorado"]
)

df2 = pd.DataFrame(
    np.arange(12.).reshape((4, 3)),
    columns=list("bde"),
    index=["Utah", "Ohio", "Texas", "Oregon"]
)

print(df1)
print()
print(df2)

            b    c    d
Ohio      0.0  1.0  2.0
Texas     3.0  4.0  5.0
Colorado  6.0  7.0  8.0

          b     d     e
Utah    0.0   1.0   2.0
Ohio    3.0   4.0   5.0
Texas   6.0   7.0   8.0
Oregon  9.0  10.0  11.0


In [6]:
df1 + df2

,b,c,d,e
Colorado,NaN,NaN,NaN,NaN
Ohio,3.0,NaN,6.0,NaN
Oregon,NaN,NaN,NaN,NaN
Texas,9.0,NaN,12.0,NaN
Utah,NaN,NaN,NaN,NaN


In [7]:
# No labels in common → entirely NaN
df1 = pd.DataFrame({"A": [1, 2]})
df2 = pd.DataFrame({"B": [3, 4]})

df1 + df2

,A,B
0,NaN,NaN
1,NaN,NaN


## Arithmetic Methods with Fill Values

When a label exists in one object but not the other, arithmetic with `+`, `-`, etc. produces `NaN`.

To substitute a specific value instead of `NaN`, use the **flexible arithmetic methods** with the `fill_value` argument:

```python
df1.add(df2, fill_value=0)
```

`fill_value` replaces missing values in **either** operand **before** the operation is applied.

### Flexible Arithmetic Methods

| Method | Operator | Reversed |
|--------|----------|----------|
| `add`, `radd` | `+` | `radd` reverses operands |
| `sub`, `rsub` | `-` | `rsub` reverses operands |
| `div`, `rdiv` | `/` | `rdiv` reverses operands |
| `floordiv`, `rfloordiv` | `//` | `rfloordiv` reverses operands |
| `mul`, `rmul` | `*` | `rmul` reverses operands |
| `pow`, `rpow` | `**` | `rpow` reverses operands |

Each method has a reversed (`r`) counterpart where the operands are swapped:

```python
1 / df1           # equivalent to:
df1.rdiv(1)
```

### `fill_value` in `reindex`

`fill_value` also works when reindexing, filling newly introduced labels with a specified value instead of `NaN`.

## Key Points

- Flexible arithmetic methods (`add`, `sub`, etc.) accept a `fill_value` argument.
- `fill_value` substitutes the given value for any `NaN` **before** the operation.
- Each method has a reversed counterpart (e.g. `rdiv`) that swaps the two operands.
- `fill_value` can also be passed to `reindex` to fill newly introduced labels.

In [8]:
df1 = pd.DataFrame(np.arange(12.).reshape((3, 4)), columns=list("abcd"))

df2 = pd.DataFrame(np.arange(20.).reshape((4, 5)), columns=list("abcde"))
df2.loc[1, "b"] = np.nan

print(df1)
print()
print(df2)

     a    b     c     d
0  0.0  1.0   2.0   3.0
1  4.0  5.0   6.0   7.0
2  8.0  9.0  10.0  11.0

      a     b     c     d     e
0   0.0   1.0   2.0   3.0   4.0
1   5.0   NaN   7.0   8.0   9.0
2  10.0  11.0  12.0  13.0  14.0
3  15.0  16.0  17.0  18.0  19.0


In [9]:
df1 + df2

,a,b,c,d,e
0,0.0,2.0,4.0,6.0,NaN
1,9.0,NaN,13.0,15.0,NaN
2,18.0,20.0,22.0,24.0,NaN
3,NaN,NaN,NaN,NaN,NaN


In [10]:
df1.add(df2, fill_value=0)

,a,b,c,d,e
0,0.0,2.0,4.0,6.0,4.0
1,9.0,5.0,13.0,15.0,9.0
2,18.0,20.0,22.0,24.0,14.0
3,15.0,16.0,17.0,18.0,19.0


In [11]:
# rdiv: equivalent to 1 / df1
df1.rdiv(1)

,a,b,c,d
0,inf,1.000000,0.500000,0.333333
1,0.250,0.200000,0.166667,0.142857
2,0.125,0.111111,0.100000,0.090909


In [12]:
df1.reindex(columns=df2.columns, fill_value=0)

,a,b,c,d,e
0,0.0,1.0,2.0,3.0,0
1,4.0,5.0,6.0,7.0,0
2,8.0,9.0,10.0,11.0,0


## Operations Between DataFrame and Series

Arithmetic between a DataFrame and a Series uses **broadcasting**, similar to NumPy.

### Default Behavior: Broadcast Down Rows

By default, pandas matches the **Series index** against the **DataFrame's column labels** and broadcasts the operation down each row:

```text
Series index  ↔  DataFrame columns
Operation repeated for every row
```

- Labels in the Series not found in the columns (or vice versa) → `NaN` for that label.

### Broadcasting Over Columns (Match on Rows)

To match the Series against the **row index** and broadcast across columns, use a flexible arithmetic method with `axis="index"`:

```python
frame.sub(series, axis="index")
```

```text
Series index  ↔  DataFrame row index
Operation repeated for every column
```

## Key Points

- DataFrame–Series arithmetic broadcasts the Series across the DataFrame.
- Default: Series index aligns to DataFrame **columns**, operation repeats down rows.
- Non-overlapping labels produce `NaN` in the result.
- To align on **rows** instead, use a flexible method with `axis="index"`.

In [13]:
# NumPy broadcasting analogy
arr = np.arange(12.).reshape((3, 4))

arr - arr[0]

array([[0., 0., 0., 0.],
       [4., 4., 4., 4.],
       [8., 8., 8., 8.]])

In [14]:
frame = pd.DataFrame(
    np.arange(12.).reshape((4, 3)),
    columns=list("bde"),
    index=["Utah", "Ohio", "Texas", "Oregon"]
)

series = frame.iloc[0]  # first row as a Series

print(frame)
print()
print(series)

          b     d     e
Utah    0.0   1.0   2.0
Ohio    3.0   4.0   5.0
Texas   6.0   7.0   8.0
Oregon  9.0  10.0  11.0

b    0.0
d    1.0
e    2.0
Name: Utah, dtype: float64


In [15]:
# Default: Series index matches columns, broadcasts down rows
frame - series

,b,d,e
Utah,0.0,0.0,0.0
Ohio,3.0,3.0,3.0
Texas,6.0,6.0,6.0
Oregon,9.0,9.0,9.0


In [16]:
# Mismatched labels → union with NaN
series2 = pd.Series(np.arange(3), index=["b", "e", "f"])

frame + series2

,b,d,e,f
Utah,0.0,NaN,3.0,NaN
Ohio,3.0,NaN,6.0,NaN
Texas,6.0,NaN,9.0,NaN
Oregon,9.0,NaN,12.0,NaN


In [17]:
series3 = frame["d"]  # a column Series, indexed by row labels

print(series3)

Utah       1.0
Ohio       4.0
Texas      7.0
Oregon    10.0
Name: d, dtype: float64


In [18]:
# axis="index": Series matches row labels, broadcasts across columns
frame.sub(series3, axis="index")

,b,d,e
Utah,-1.0,0.0,1.0
Ohio,-1.0,0.0,1.0
Texas,-1.0,0.0,1.0
Oregon,-1.0,0.0,1.0
